In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.examples.tutorials.mnist import input_data

# 载入数据集
mnist = input_data.read_data_sets("MNIST_data", one_hot=True)
tf.device('/gpu:1')

D:\Users\wzq11\anaconda3\envs\tensorflow-gpu-1.8.0\lib\site-packages\tensorflow\python\framework\dtypes.py:519: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
D:\Users\wzq11\anaconda3\envs\tensorflow-gpu-1.8.0\lib\site-packages\tensorflow\python\framework\dtypes.py:520: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
D:\Users\wzq11\anaconda3\envs\tensorflow-gpu-1.8.0\lib\site-packages\tensorflow\python\framework\dtypes.py:521: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
D:\Users\wzq11\anaconda3\env

Instructions for updating:
Please use alternatives such as official/mnist/dataset.py from tensorflow/models.
Instructions for updating:
Please write your own downloading logic.
Instructions for updating:
Please use tf.data to implement this functionality.
Extracting MNIST_data\train-images-idx3-ubyte.gz
Instructions for updating:
Please use tf.data to implement this functionality.
Extracting MNIST_data\train-labels-idx1-ubyte.gz
Instructions for updating:
Please use tf.one_hot on tensors.
Extracting MNIST_data\t10k-images-idx3-ubyte.gz
Extracting MNIST_data\t10k-labels-idx1-ubyte.gz
Instructions for updating:
Please use alternatives such as official/mnist/dataset.py from tensorflow/models.


In [9]:
n_epochs = 100  # 训练总轮次
batch_size = 100  # 每个批次大小
# 计算一共有多少个批次
n_batch = mnist.train.num_examples // batch_size  # 总样本数整除每个批次大小

# 定义两个placeholder
x = tf.placeholder(tf.float32, [None, 784])  # 占位符，x用作输入变量
y = tf.placeholder(tf.float32, [None, 10])  # 占位符，y代表真实标签
keep_prob = tf.placeholder(tf.float32)  # 定义使用的神经元占该层神经元总数的比例
lr = tf.Variable(0.001, dtype=tf.float32)  # 学习速率

#创建一个简单的神经网络
#更为常见的初始化方式，高斯分布
W1 = tf.Variable(tf.truncated_normal([784,600],stddev=0.1))  # 初始化权重W1
b1 = tf.Variable(tf.zeros([600])+0.1)  # 初始化偏置b1
# 激活层
L1 = tf.nn.relu(tf.matmul(x, W1)+b1)  # 以relu函数激活
# drop层
L1_drop = tf.nn.dropout(L1,keep_prob)

W2 = tf.Variable(tf.truncated_normal([600,400],stddev=0.1))
b2 = tf.Variable(tf.zeros([400])+0.1)
L2 = tf.nn.relu(tf.matmul(L1_drop, W2)+b2) 
L2_drop = tf.nn.dropout(L2,keep_prob)

W3 = tf.Variable(tf.truncated_normal([400,10],stddev=0.1))
b3 = tf.Variable(tf.zeros([10])+0.1)
prediction = tf.nn.softmax(tf.matmul(L2_drop,W3) + b3)

#交叉熵代价函数
loss = tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits_v2(logits=prediction,labels=y))
train_step = tf.train.AdamOptimizer(lr).minimize(loss)  # Adam优化器最小化损失函数，反向传播更新权重
    

init = tf.global_variables_initializer()

# one_hot编码结果存放到一个布尔型表中
correct_prediction = tf.equal(tf.argmax(y, 1), tf.argmax(prediction, 1))
# 求准确率
accuracy = tf.reduce_mean(tf.cast(correct_prediction, tf.float32))

with tf.Session() as sess:
    sess.run(init)
    for epoch in range(n_epochs):
        sess.run(tf.assign(lr, 0.001 * (0.98 ** epoch)))
        for batch in range(n_batch):
            batch_xs,batch_ys = mnist.train.next_batch(batch_size)
            sess.run(train_step, feed_dict={x:batch_xs, y:batch_ys,keep_prob:0.6})
            
        learning_rate = sess.run(lr)
        test_acc = sess.run(accuracy, feed_dict={x:mnist.test.images, y:mnist.test.labels,keep_prob:1})
        train_acc = sess.run(accuracy, feed_dict={x:mnist.train.images, y:mnist.train.labels,keep_prob:0.8})
        print("Epoch:" + str(epoch+1) + ",Testing Accuracy:" + str(test_acc)+ ",Training Accuracy:" + str(train_acc)+",Learning Rate:"+str(learning_rate))

Epoch:1,Testing Accuracy:0.939,Training Accuracy:0.93265456,Learning Rate:0.001
Epoch:2,Testing Accuracy:0.9574,Training Accuracy:0.9557273,Learning Rate:0.00098
Epoch:3,Testing Accuracy:0.9648,Training Accuracy:0.9614546,Learning Rate:0.0009604
Epoch:4,Testing Accuracy:0.9663,Training Accuracy:0.9662182,Learning Rate:0.000941192
Epoch:5,Testing Accuracy:0.969,Training Accuracy:0.97025454,Learning Rate:0.00092236814
Epoch:6,Testing Accuracy:0.9692,Training Accuracy:0.97043633,Learning Rate:0.0009039208
Epoch:7,Testing Accuracy:0.9696,Training Accuracy:0.9721091,Learning Rate:0.00088584237
Epoch:8,Testing Accuracy:0.9745,Training Accuracy:0.9774182,Learning Rate:0.0008681255
Epoch:9,Testing Accuracy:0.975,Training Accuracy:0.9770909,Learning Rate:0.000850763
Epoch:10,Testing Accuracy:0.9758,Training Accuracy:0.9789091,Learning Rate:0.00083374773
Epoch:11,Testing Accuracy:0.9746,Training Accuracy:0.97827274,Learning Rate:0.0008170728
Epoch:12,Testing Accuracy:0.9759,Training Accuracy:0.9

Epoch:94,Testing Accuracy:0.9863,Training Accuracy:0.9967273,Learning Rate:0.00015276541
Epoch:95,Testing Accuracy:0.9858,Training Accuracy:0.9966,Learning Rate:0.0001497101
Epoch:96,Testing Accuracy:0.9862,Training Accuracy:0.99654543,Learning Rate:0.0001467159
Epoch:97,Testing Accuracy:0.986,Training Accuracy:0.9966364,Learning Rate:0.00014378158
Epoch:98,Testing Accuracy:0.9865,Training Accuracy:0.99674547,Learning Rate:0.00014090595
Epoch:99,Testing Accuracy:0.986,Training Accuracy:0.99667275,Learning Rate:0.00013808784
Epoch:100,Testing Accuracy:0.9863,Training Accuracy:0.99683636,Learning Rate:0.00013532607
